
# NASA CMAPSS FD002 — 최고점 도전용 Synergy Notebook (Cluster-Norm + Seq DL + Optuna HPO + Ensemble)

이 노트북은 Kaggle의 NASA CMAPSS **FD002** 데이터셋에서 점수(특히 NASA score / RMSE 개선)를 최대한 끌어올리기 위한 “올인원” 파이프라인입니다.

구성:
- 데이터 로드 & EDA (분포/상관/저분산/클러스터/PCA 등)
- RUL 생성 + 캡(capping)
- **운영조건(op1~3) 기반 KMeans 클러스터링**
- **클러스터별 정규화(Cluster-wise StandardScaler)** + (옵션) EWMA/rolling features
- 시퀀스 데이터셋 생성(windowing)
- 모델: **CNN + BiLSTM + Attention**
- 평가: RMSE/MAE + **NASA score**
- **Optuna 하이퍼파라미터 튜닝(전처리 + 모델 + 학습)** + Pruning
- Best params로 재학습 + (옵션) **Multi-seed Ensemble**
- Test 예측 및 submission 생성
- 시각화: 학습곡선/잔차/유닛별 예측/Attention/Optuna plots

> Kaggle 환경 기준 경로로 작성되어 있습니다. 로컬 실행 시 `DATA_DIR`만 수정하세요.


In [ ]:

# =====================
# 0) 환경 설정
# =====================
import os, gc, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
seed_everything(SEED)

print("OK")


In [ ]:

# (선택) Optuna가 없으면 설치
import importlib, sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import optuna
    print("optuna:", optuna.__version__)
except Exception as e:
    print("Installing optuna...")
    pip_install("optuna>=3.6.0")
    import optuna
    print("optuna:", optuna.__version__)


In [ ]:

# ML / DL
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print("TF:", tf.__version__)



## 1) 데이터 로드

Kaggle 데이터셋: `behrad3d/nasa-cmaps`  
이 노트북은 **FD002**만 사용하도록 기본 설정되어 있습니다.


In [ ]:

# =====================
# 1) 데이터 로드
# =====================
DATA_DIR = "/kaggle/input/nasa-cmaps"  # Kaggle
# 로컬이면 예: DATA_DIR = "./data/nasa-cmaps"

# CMAPSS 컬럼명
sensor_cols = [f"s{i}" for i in range(1, 22)]
cols = ["unit", "cycle", "op1", "op2", "op3"] + sensor_cols

def load_split(fd="FD002"):
    train_path = os.path.join(DATA_DIR, f"train_{fd}.txt")
    test_path  = os.path.join(DATA_DIR, f"test_{fd}.txt")
    rul_path   = os.path.join(DATA_DIR, f"RUL_{fd}.txt")

    train = pd.read_csv(train_path, sep=r"\s+", header=None)
    test  = pd.read_csv(test_path,  sep=r"\s+", header=None)
    rul   = pd.read_csv(rul_path,   sep=r"\s+", header=None)

    train = train.iloc[:, :len(cols)]
    test  = test.iloc[:, :len(cols)]
    train.columns = cols
    test.columns  = cols

    rul = rul.iloc[:, 0].values.astype(int)
    return train, test, rul

train_df, test_df, rul_test = load_split("FD002")

print(train_df.shape, test_df.shape, rul_test.shape)
train_df.head()



## 2) EDA & 기본 전처리

- unit별 cycle 길이 분포
- 센서 저분산 탐지
- 상관관계 히트맵
- 운영조건(op1~3) 기반 클러스터링 개요


In [ ]:

# unit별 최대 cycle 분포
max_cycle = train_df.groupby("unit")["cycle"].max()
plt.figure()
plt.hist(max_cycle.values, bins=30)
plt.title("Train units: max cycle distribution")
plt.xlabel("max cycle"); plt.ylabel("count")
plt.show()

print(max_cycle.describe())


In [ ]:

# 저분산 센서 확인
sensor_var = train_df[sensor_cols].var().sort_values()
plt.figure()
plt.bar(range(len(sensor_var)), sensor_var.values)
plt.title("Sensor variance (ascending)")
plt.xlabel("sensor (sorted)"); plt.ylabel("variance")
plt.show()

sensor_var.head(10)


In [ ]:

# 상관관계 히트맵 (샘플링으로 속도 개선)
sample = train_df.sample(min(5000, len(train_df)), random_state=SEED)
corr = sample[sensor_cols].corr()

plt.figure(figsize=(10,8))
plt.imshow(corr, aspect="auto")
plt.title("Sensor correlation (sample)")
plt.colorbar()
plt.show()



## 3) RUL 생성 + 캡(capping)

Train: 각 unit의 최대 cycle을 기반으로 RUL 생성.  
Test: 제공된 `RUL_FD002.txt`로 마지막 시점의 RUL이 주어짐(Leaderboard용).


In [ ]:

def add_rul(train):
    max_cycle = train.groupby("unit")["cycle"].max().rename("max_cycle")
    df = train.merge(max_cycle, on="unit", how="left")
    df["RUL"] = df["max_cycle"] - df["cycle"]
    df.drop(columns=["max_cycle"], inplace=True)
    return df

train_df = add_rul(train_df)

train_df[["unit","cycle","RUL"]].head()


In [ ]:

plt.figure()
plt.hist(train_df["RUL"].values, bins=60)
plt.title("RUL distribution (raw)")
plt.xlabel("RUL"); plt.ylabel("count")
plt.show()

print(train_df["RUL"].describe())



## 4) NASA score (평가 지표)

Kaggle 리더보드에서 주로 쓰이는 점수는 RMSE 기반이지만, NASA 논문/기존 CMAPSS 기준으로는 아래 **asymmetric** score도 자주 사용합니다.
- under-estimate(빨리 고장 예측) vs over-estimate(늦게 고장 예측) 페널티가 다름


In [ ]:

def nasa_score(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    d = y_pred - y_true
    score = 0.0
    for di in d:
        if di < 0:
            score += math.exp(-di/13.0) - 1.0
        else:
            score += math.exp(di/10.0) - 1.0
    return score

# quick sanity
print(nasa_score([10,10],[10,10]), nasa_score([10,10],[0,0]), nasa_score([10,10],[20,20]))



## 5) 핵심: 운영조건 기반 클러스터링 + 클러스터별 정규화

FD002는 운영조건이 여러 모드로 섞여 있어, 전체 스케일러를 한 번에 적용하면 성능이 흔들리는 경우가 많습니다.

절차:
1. Train의 op1~3로 KMeans 클러스터 학습
2. 각 클러스터별로 센서 스케일러(StandardScaler)를 따로 fit
3. Train/Test 각각에서 (op→cluster) 할당 후 해당 클러스터 스케일러로 변환


In [ ]:

def make_clusters(train, test, n_clusters, seed=SEED):
    km = KMeans(n_clusters=n_clusters, random_state=seed, n_init="auto")
    km.fit(train[["op1","op2","op3"]])
    train_c = km.predict(train[["op1","op2","op3"]])
    test_c  = km.predict(test[["op1","op2","op3"]])
    return km, train_c, test_c

def clusterwise_scale(train, test, train_cluster, test_cluster, feature_cols):
    train_scaled = train.copy()
    test_scaled  = test.copy()
    scalers = {}
    for c in np.unique(train_cluster):
        sc = StandardScaler()
        idx_tr = (train_cluster==c)
        sc.fit(train.loc[idx_tr, feature_cols])
        train_scaled.loc[idx_tr, feature_cols] = sc.transform(train.loc[idx_tr, feature_cols])
        idx_te = (test_cluster==c)
        test_scaled.loc[idx_te, feature_cols]  = sc.transform(test.loc[idx_te, feature_cols])
        scalers[int(c)] = sc
    return train_scaled, test_scaled, scalers

# 빠른 시각화: op 공간에서 PCA 2D
pca = PCA(n_components=2, random_state=SEED)
op_2d = pca.fit_transform(train_df[["op1","op2","op3"]].values)
plt.figure()
plt.scatter(op_2d[:,0], op_2d[:,1], s=2, alpha=0.3)
plt.title("Operating condition PCA (train)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()



## 6) Feature engineering 옵션 (튜닝 대상)

- **EWMA smoothing**: 센서 노이즈 완화 (alpha 튜닝)
- **rolling mean/std**: 작은 윈도우로 추세/변동성 특징 추가 (window 튜닝)
- **저분산 센서 제거**: var threshold 튜닝


In [ ]:

def add_ewma(df, cols, alpha):
    out = df.copy()
    if alpha is None:
        return out
    out = out.sort_values(["unit","cycle"])
    for c in cols:
        out[c] = out.groupby("unit")[c].transform(lambda x: x.ewm(alpha=alpha, adjust=False).mean())
    return out

def add_rolling(df, cols, window):
    out = df.copy()
    if window is None or window <= 1:
        return out
    out = out.sort_values(["unit","cycle"])
    for c in cols:
        out[f"{c}_rm{window}"] = out.groupby("unit")[c].transform(lambda x: x.rolling(window, min_periods=1).mean())
        out[f"{c}_rs{window}"] = out.groupby("unit")[c].transform(lambda x: x.rolling(window, min_periods=1).std().fillna(0.0))
    return out

def drop_low_variance(train, test, sensor_cols, var_thr):
    if var_thr is None or var_thr <= 0:
        return train, test, sensor_cols
    v = train[sensor_cols].var()
    keep = v[v > var_thr].index.tolist()
    return train, test, keep



## 7) 시퀀스 데이터셋 생성

- window 길이 `seq_len`
- stride `step`
- Train: 모든 가능한 윈도우 생성(각 윈도우의 라벨은 윈도우 마지막 시점의 RUL)
- Test: 각 unit의 **마지막 윈도우**만 사용(리더보드 제출용)


In [ ]:

def make_windows(df, feature_cols, seq_len=30, step=1, label_col="RUL", is_train=True):
    df = df.sort_values(["unit","cycle"])
    X, y, groups = [], [], []
    for u, g in df.groupby("unit"):
        g = g.reset_index(drop=True)
        feats = g[feature_cols].values.astype(np.float32)
        if is_train:
            labels = g[label_col].values.astype(np.float32)
            for start in range(0, len(g)-seq_len+1, step):
                end = start + seq_len
                X.append(feats[start:end])
                y.append(labels[end-1])
                groups.append(u)
        else:
            # last window
            if len(g) < seq_len:
                pad = np.zeros((seq_len - len(g), feats.shape[1]), dtype=np.float32)
                window = np.vstack([pad, feats])
            else:
                window = feats[-seq_len:]
            X.append(window)
            groups.append(u)
    X = np.stack(X)
    if is_train:
        y = np.asarray(y).reshape(-1,1)
        groups = np.asarray(groups)
        return X, y, groups
    else:
        groups = np.asarray(groups)
        return X, groups

print("OK")



## 8) 모델: CNN + BiLSTM + Attention

튜닝 대상:
- conv filters / kernel
- lstm units
- dense units
- dropout
- learning rate


In [ ]:

class AttentionLayer(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], 1), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(1,), initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, x):
        # x: (B, T, C)
        e = tf.tensordot(x, self.W, axes=1) + self.b  # (B,T,1)
        a = tf.nn.softmax(e, axis=1)                  # (B,T,1)
        context = tf.reduce_sum(x * a, axis=1)        # (B,C)
        return context, tf.squeeze(a, axis=-1)        # (B,C), (B,T)

def build_model(input_shape, hp):
    inp = keras.Input(shape=input_shape)
    x = inp
    x = layers.Conv1D(filters=hp["conv_filters"], kernel_size=hp["conv_kernel"], padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(hp["dropout"])(x)

    x = layers.Bidirectional(layers.LSTM(hp["lstm_units"], return_sequences=True))(x)
    x = layers.Dropout(hp["dropout"])(x)

    context, att = AttentionLayer(name="attention")(x)
    x = layers.Dense(hp["dense_units"], activation="relu")(context)
    x = layers.Dropout(hp["dropout"])(x)
    out = layers.Dense(1, activation="linear")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp["lr"]),
        loss="mse",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse"), keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model



## 9) Train/Valid split (unit 단위 group split)

중요: 시계열/유닛 데이터는 랜덤 샘플 split하면 leakage가 생길 수 있어서, **unit 단위로 split**합니다.


In [ ]:

def group_train_valid_split(train_df, test_size=0.2, seed=SEED):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx = np.arange(len(train_df))
    (tr_idx, va_idx) = next(gss.split(idx, groups=train_df["unit"].values))
    return train_df.iloc[tr_idx].copy(), train_df.iloc[va_idx].copy()

print("OK")



## 10) Optuna 하이퍼파라미터 튜닝

튜닝 범위(권장 “최고 단계”):
- rul_cap
- n_clusters
- seq_len / step
- EWMA alpha
- rolling window
- var threshold
- model 구조 & 학습률 & batch size
- 학습 epoch (튜닝은 짧게, 최종은 길게)

> 튜닝은 시간이 걸립니다. Kaggle GPU에서 `n_trials`를 30~80 정도 권장(시간 여유가 있으면 150+).


In [ ]:

def prepare_features(train_df, test_df, hp, seed=SEED):
    # 1) RUL cap
    rul_cap = hp["rul_cap"]
    tr = train_df.copy()
    tr["RUL"] = tr["RUL"].clip(0, rul_cap)

    te = test_df.copy()

    # 2) low variance drop (sensor only 기준)
    var_thr = hp["var_thr"]
    tr2, te2, kept_sensors = drop_low_variance(tr, te, sensor_cols, var_thr)

    # 3) optional smoothing / rolling (sensor columns 기준, 이후 파생칼럼 포함)
    tr2 = add_ewma(tr2, kept_sensors, hp["ewma_alpha"])
    te2 = add_ewma(te2, kept_sensors, hp["ewma_alpha"])

    tr2 = add_rolling(tr2, kept_sensors, hp["roll_w"])
    te2 = add_rolling(te2, kept_sensors, hp["roll_w"])

    # 업데이트된 feature columns
    feature_cols = [c for c in tr2.columns if (c in kept_sensors) or (c.startswith("s") and ("_rm" in c or "_rs" in c))]
    feature_cols = sorted(feature_cols)  # 안정적 순서

    # 4) clustering on ops
    km, tr_c, te_c = make_clusters(tr2, te2, hp["n_clusters"], seed=seed)

    # 5) cluster-wise scaling
    tr_scaled, te_scaled, _ = clusterwise_scale(tr2, te2, tr_c, te_c, feature_cols)

    tr_scaled["cluster"] = tr_c
    te_scaled["cluster"] = te_c
    return tr_scaled, te_scaled, feature_cols, km

def train_one_trial(train_df, test_df, hp, seed=SEED, verbose=0):
    # unit-wise split
    tr_u, va_u = group_train_valid_split(train_df, test_size=0.2, seed=seed)

    tr_scaled, _, feat_cols, _ = prepare_features(tr_u, test_df, hp, seed=seed)
    va_scaled, _, _, _ = prepare_features(va_u, test_df, hp, seed=seed)  # fit을 train/valid 각각 하면 안 되나?
    # ⚠️ 주의: 위처럼 valid에서 다시 fit하면 leakage/불일치 가능.
    # 그래서 실제로는 train 기반으로만 클러스터+스케일러를 fit하고 valid/test에 transform해야 함.
    # 아래에서 이를 올바르게 구현한 파이프라인 사용.

    return None



### 10-1) 올바른 파이프라인: Train 기반 fit → Valid/Test transform

튜닝에서도 반드시 **fit은 train split에만**, valid/test는 transform만 해야 안정적입니다.


In [ ]:

def fit_preprocessor(train_split, hp, seed=SEED):
    # RUL cap
    tr = train_split.copy()
    tr["RUL"] = tr["RUL"].clip(0, hp["rul_cap"])

    # low variance sensor filter
    kept = sensor_cols
    if hp["var_thr"] and hp["var_thr"] > 0:
        v = tr[kept].var()
        kept = v[v > hp["var_thr"]].index.tolist()

    # smoothing / rolling on train
    tr = add_ewma(tr, kept, hp["ewma_alpha"])
    tr = add_rolling(tr, kept, hp["roll_w"])

    # feature columns include derived
    feat_cols = [c for c in tr.columns if (c in kept) or (c.startswith("s") and ("_rm" in c or "_rs" in c))]
    feat_cols = sorted(feat_cols)

    # cluster fit on ops
    km = KMeans(n_clusters=hp["n_clusters"], random_state=seed, n_init="auto")
    km.fit(tr[["op1","op2","op3"]])
    tr_c = km.predict(tr[["op1","op2","op3"]])

    # cluster-wise scalers fit
    scalers = {}
    for c in np.unique(tr_c):
        sc = StandardScaler()
        sc.fit(tr.loc[tr_c==c, feat_cols])
        scalers[int(c)] = sc

    return {"km": km, "scalers": scalers, "feat_cols": feat_cols, "kept_sensors": kept}

def transform_with_preprocessor(df, prep, hp):
    out = df.copy()

    # RUL cap if exists
    if "RUL" in out.columns:
        out["RUL"] = out["RUL"].clip(0, hp["rul_cap"])

    # apply smoothing/rolling using kept sensors
    out = add_ewma(out, prep["kept_sensors"], hp["ewma_alpha"])
    out = add_rolling(out, prep["kept_sensors"], hp["roll_w"])

    feat_cols = prep["feat_cols"]

    # cluster assign
    c = prep["km"].predict(out[["op1","op2","op3"]])
    out["cluster"] = c

    # scale per cluster
    for cl, sc in prep["scalers"].items():
        idx = (out["cluster"].values == cl)
        if idx.any():
            out.loc[idx, feat_cols] = sc.transform(out.loc[idx, feat_cols])

    return out

def build_datasets(train_df, valid_df, test_df, prep, hp):
    tr_t = transform_with_preprocessor(train_df, prep, hp)
    va_t = transform_with_preprocessor(valid_df, prep, hp)
    te_t = transform_with_preprocessor(test_df,  prep, hp)

    X_tr, y_tr, g_tr = make_windows(tr_t, prep["feat_cols"], seq_len=hp["seq_len"], step=hp["step"], is_train=True)
    X_va, y_va, g_va = make_windows(va_t, prep["feat_cols"], seq_len=hp["seq_len"], step=hp["step"], is_train=True)
    X_te, g_te      = make_windows(te_t, prep["feat_cols"], seq_len=hp["seq_len"], step=1, is_train=False)
    return X_tr, y_tr, g_tr, X_va, y_va, g_va, X_te, g_te

print("OK")


In [ ]:

def objective(trial):
    # ===== search space =====
    hp = {
        # preprocessing
        "rul_cap": trial.suggest_int("rul_cap", 100, 150),
        "n_clusters": trial.suggest_int("n_clusters", 2, 8),
        "seq_len": trial.suggest_int("seq_len", 20, 60),
        "step": trial.suggest_int("step", 1, 5),
        "ewma_alpha": trial.suggest_float("ewma_alpha", 0.05, 0.4),
        "roll_w": trial.suggest_int("roll_w", 1, 9),
        "var_thr": trial.suggest_float("var_thr", 0.0, 1e-3),

        # model
        "conv_filters": trial.suggest_categorical("conv_filters", [32, 48, 64, 96, 128]),
        "conv_kernel": trial.suggest_categorical("conv_kernel", [3, 5, 7]),
        "lstm_units": trial.suggest_categorical("lstm_units", [32, 48, 64, 96, 128]),
        "dense_units": trial.suggest_categorical("dense_units", [32, 64, 96, 128, 192]),
        "dropout": trial.suggest_float("dropout", 0.05, 0.4),
        "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "batch": trial.suggest_categorical("batch", [64, 128, 256]),
    }

    # ===== unit-wise split =====
    train_units = train_df["unit"].unique()
    rng = np.random.RandomState(SEED)
    rng.shuffle(train_units)
    n_val = max(1, int(0.2 * len(train_units)))
    val_units = set(train_units[:n_val])
    tr_split = train_df[~train_df["unit"].isin(val_units)].copy()
    va_split = train_df[ train_df["unit"].isin(val_units)].copy()

    # ===== fit preprocessor on train split only =====
    prep = fit_preprocessor(tr_split, hp, seed=SEED)

    # ===== build windows =====
    X_tr, y_tr, _, X_va, y_va, _, _, _ = build_datasets(tr_split, va_split, test_df, prep, hp)

    # sanity / speed guard
    if X_tr.shape[0] < 2000:
        raise optuna.TrialPruned()

    # ===== model =====
    model = build_model(X_tr.shape[1:], hp)

    cb = [
        keras.callbacks.EarlyStopping(monitor="val_rmse", patience=5, mode="min", restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_rmse", factor=0.5, patience=2, min_lr=1e-6, mode="min"),
        optuna.integration.TFKerasPruningCallback(trial, monitor="val_rmse"),
    ]

    hist = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=25,  # tuning epochs (짧게)
        batch_size=hp["batch"],
        verbose=0,
        callbacks=cb
    )

    # evaluate
    va_pred = model.predict(X_va, batch_size=hp["batch"], verbose=0).reshape(-1)
    va_true = y_va.reshape(-1)

    rmse = math.sqrt(mean_squared_error(va_true, va_pred))
    # 보너스: NASA score도 같이 참고 가능 (튜닝 목적함수는 RMSE로)
    trial.set_user_attr("nasa_score", float(nasa_score(va_true, va_pred)))
    return rmse


In [ ]:

# =====================
# Optuna 실행
# =====================
import optuna
from optuna.samplers import TPESampler

N_TRIALS = 40   # 최고점 목표면 80~200+ 권장 (시간/자원에 맞게 조절)
study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)
print("Best trial NASA score:", study.best_trial.user_attrs.get("nasa_score"))



### 10-2) Optuna 시각화

Kaggle 노트북에서 plotly가 기본 제공됩니다(안 되면 pip install plotly).


In [ ]:

try:
    import optuna.visualization as vis
    fig1 = vis.plot_optimization_history(study)
    fig2 = vis.plot_param_importances(study)
    fig1.show()
    fig2.show()
except Exception as e:
    print("Optuna visualization error:", e)



## 11) Best params로 최종 학습 (길게) + Multi-seed Ensemble (선택)

- Optuna는 빠르게 탐색하기 위해 epoch을 짧게 둡니다.
- Best params로 **epoch을 늘려** 재학습하면 점수가 더 좋아지는 경우가 많습니다.
- 추가로 seed를 여러 개로 학습해서 평균을 내면 분산이 줄어 점수가 더 좋아질 수 있습니다.


In [ ]:

best_hp = study.best_params.copy()

# 튜닝에서 사용한 파라미터들에 누락이 없도록 기본값 병합
defaults = dict(
    rul_cap=125, n_clusters=4, seq_len=30, step=1,
    ewma_alpha=0.2, roll_w=3, var_thr=0.0,
    conv_filters=64, conv_kernel=5, lstm_units=64, dense_units=96,
    dropout=0.2, lr=1e-3, batch=128
)
for k,v in defaults.items():
    best_hp.setdefault(k,v)

best_hp


In [ ]:

def train_full_and_predict(train_df, test_df, hp, seed=SEED, epochs=80, verbose=0):
    # train/valid split by units for monitoring (하지만 최종은 train 전체로 학습하는 것이 일반적)
    # 여기서는 early stopping 관리를 위해 작은 valid를 두고, best weights로 최종 예측
    train_units = train_df["unit"].unique()
    rng = np.random.RandomState(seed)
    rng.shuffle(train_units)
    n_val = max(1, int(0.15 * len(train_units)))
    val_units = set(train_units[:n_val])
    tr_split = train_df[~train_df["unit"].isin(val_units)].copy()
    va_split = train_df[ train_df["unit"].isin(val_units)].copy()

    prep = fit_preprocessor(tr_split, hp, seed=seed)
    X_tr, y_tr, _, X_va, y_va, _, X_te, g_te = build_datasets(tr_split, va_split, test_df, prep, hp)

    model = build_model(X_tr.shape[1:], hp)
    cb = [
        keras.callbacks.EarlyStopping(monitor="val_rmse", patience=10, mode="min", restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_rmse", factor=0.5, patience=3, min_lr=1e-6, mode="min"),
    ]
    hist = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=epochs,
        batch_size=hp["batch"],
        verbose=verbose,
        callbacks=cb
    )

    # valid eval
    va_pred = model.predict(X_va, batch_size=hp["batch"], verbose=0).reshape(-1)
    va_true = y_va.reshape(-1)
    rmse = math.sqrt(mean_squared_error(va_true, va_pred))
    mae  = mean_absolute_error(va_true, va_pred)
    ns   = nasa_score(va_true, va_pred)

    # test predict (last window per unit)
    te_pred = model.predict(X_te, batch_size=hp["batch"], verbose=0).reshape(-1)

    return {
        "model": model,
        "history": hist.history,
        "val_rmse": rmse,
        "val_mae": mae,
        "val_nasa": ns,
        "test_pred": te_pred,
        "test_units": g_te,
        "prep": prep
    }

res = train_full_and_predict(train_df, test_df, best_hp, seed=SEED, epochs=100, verbose=0)
print("VAL RMSE:", res["val_rmse"], "MAE:", res["val_mae"], "NASA:", res["val_nasa"])


In [ ]:

# 학습 곡선 시각화
hist = res["history"]
plt.figure()
plt.plot(hist["rmse"], label="train_rmse")
plt.plot(hist["val_rmse"], label="val_rmse")
plt.title("Learning curve (RMSE)")
plt.xlabel("epoch"); plt.ylabel("rmse")
plt.legend()
plt.show()

plt.figure()
plt.plot(hist["loss"], label="train_loss")
plt.plot(hist["val_loss"], label="val_loss")
plt.title("Learning curve (Loss)")
plt.xlabel("epoch"); plt.ylabel("mse")
plt.legend()
plt.show()



### (선택) Multi-seed Ensemble

가끔 leaderboard에서 꽤 먹히는 방법입니다. (단, 시간/자원 증가)


In [ ]:

ENSEMBLE = True
SEEDS = [11, 22, 33]  # 더 늘리면 성능↑ 가능, 시간↑↑
epochs = 120

if ENSEMBLE:
    preds = []
    metrics = []
    for sd in SEEDS:
        seed_everything(sd)
        r = train_full_and_predict(train_df, test_df, best_hp, seed=sd, epochs=epochs, verbose=0)
        preds.append(r["test_pred"])
        metrics.append((sd, r["val_rmse"], r["val_mae"], r["val_nasa"]))
        print("seed", sd, "val_rmse", r["val_rmse"], "val_nasa", r["val_nasa"])
    pred_ens = np.mean(np.stack(preds), axis=0)
    print("Ensemble done. shape:", pred_ens.shape)
else:
    pred_ens = res["test_pred"]


In [ ]:

# Submission 생성
# test_units 는 unit 순서(각 unit 마지막 window)
sub = pd.DataFrame({"unit": (res["test_units"] if not ENSEMBLE else res["test_units"]),
                    "RUL": pred_ens})
sub = sub.sort_values("unit").reset_index(drop=True)

# 예측 안정성: 음수 방지 + cap 적용
sub["RUL"] = sub["RUL"].clip(0, best_hp["rul_cap"])

sub.head(), sub.shape


In [ ]:

# Kaggle 제출용 파일 저장
sub_path = "submission.csv"
sub[["RUL"]].to_csv(sub_path, index=False)
print("Saved:", sub_path)



## 12) 추가 시각화: 유닛별 예측 vs 정답(테스트 라스트 RUL)

FD002 test는 마지막 시점의 RUL 정답이 제공되므로, 간단 검증 플롯을 그릴 수 있습니다.


In [ ]:

# test 정답(rul_test)은 unit 순서대로 주어진다고 가정 (CMAPSS 규칙)
y_test_true = np.asarray(rul_test).reshape(-1)
y_test_pred = sub["RUL"].values.reshape(-1)

rmse_test = math.sqrt(mean_squared_error(y_test_true, y_test_pred))
mae_test  = mean_absolute_error(y_test_true, y_test_pred)
ns_test   = nasa_score(y_test_true, y_test_pred)
print("Test(last-cycle) RMSE:", rmse_test, "MAE:", mae_test, "NASA:", ns_test)

plt.figure()
plt.scatter(y_test_true, y_test_pred, s=10, alpha=0.6)
plt.title("Test: True vs Pred (last-cycle per unit)")
plt.xlabel("True RUL"); plt.ylabel("Pred RUL")
plt.plot([0, best_hp["rul_cap"]], [0, best_hp["rul_cap"]])
plt.show()

residual = y_test_pred - y_test_true
plt.figure()
plt.hist(residual, bins=40)
plt.title("Residual distribution (Pred - True)")
plt.xlabel("residual"); plt.ylabel("count")
plt.show()



## 13) Attention 시각화 (샘플 유닛)

Attention이 어떤 타임스텝을 더 중요하게 보는지 확인합니다.


In [ ]:

# Attention weight 추출: attention layer 출력은 (context, att_weights)지만 model 출력은 RUL만.
# 그래서 attention weights를 뽑는 별도 모델을 구성합니다.

def make_attention_extractor(model):
    att_layer = model.get_layer("attention")
    # attention layer는 (context, att) 튜플을 반환하므로, 그 중 att를 출력하도록 구성
    # 모델 그래프에서 attention layer의 output은 [context, att]
    att_out = att_layer.output[1]
    return keras.Model(model.input, att_out)

att_model = make_attention_extractor(res["model"])
print(att_model.summary())


In [ ]:

# test window 중 하나 뽑아서 attention plot
# (ENSEMBLE일 때는 마지막으로 학습한 res["model"] 기준)
X_te, g_te = make_windows(transform_with_preprocessor(test_df, res["prep"], best_hp),
                          res["prep"]["feat_cols"], seq_len=best_hp["seq_len"], is_train=False)

idx = 0
att_w = att_model.predict(X_te[idx:idx+1], verbose=0).reshape(-1)

plt.figure()
plt.plot(att_w)
plt.title(f"Attention weights (unit={g_te[idx]})")
plt.xlabel("time step"); plt.ylabel("weight")
plt.show()



## 14) 최고점 팁 (실전)

- `N_TRIALS`를 늘리세요 (80~200+)
- `seq_len` 탐색 범위를 넓혀보세요 (예: 20~90)
- `n_clusters`를 2~12로 넓혀보세요
- rolling 특징을 더 늘리려면:
  - slope(선형회귀 기울기), delta(차분), min/max 등 파생 추가
- 모델을 2개 계열로 앙상블하면 더 좋아질 수 있습니다:
  - (A) CNN+BiLSTM+Attention (현재)
  - (B) Transformer encoder 기반 시계열 모델
- **submission 안정화**: 예측을 clip/rounding/median ensemble 등으로 튜닝

필요하면 Transformer 버전까지 포함한 **2-모델 앙상블 최고점 노트북**도 만들어 드릴게요.
